## GENeSYS-MOD to IAMC conversion

Converts GENeSYS-MOD results into the IAMC format of the IIASA scenario explorers and checks the upload file with the explorer's own upload workflow.

1. Put the results of every scenario into `genesysmod_to_iamc/input/`: `<scenario>.gdx` of the GAMS version, or the result csv files of the Julia version (written with `switch_processed_results = 1`).
2. Choose the settings in the settings cells below.
3. Run all cells (*Run → Run All Cells*).

The notebook needs the Python environment of `environment.yml`. Create it once with `conda env create -f environment.yml`, then either start Jupyter in it (`conda activate genesysmod-iamc`, then `jupyter notebook` in this folder) or choose `genesysmod-iamc` as the kernel in VS Code.

Without Jupyter, the same conversion runs from a terminal with `python run.py`, which has its own copy of these settings at its top. `README.md` explains both ways, the conversion and its assumptions.

#### Where to change what

- which GENeSYS-MOD technology or fuel goes to which IAMC variable: the csv files in `genesysmod_to_iamc/mappings/` (table in `README.md`)
- the sums of variables and the total over all regions: `genesysmod_to_iamc/mappings/aggregations.yaml`
- which variables, regions and units the explorer accepts: the explorer's workflow repository, not this one

Changes to the mapping files apply at the next run. After changes to the code in `genesysmod_to_iamc/`, restart the kernel.

In [ ]:
import genesysmod_to_iamc

### Settings

Everything that changes between releases or projects, in four cells. Folders are relative to the folder of this notebook. After a change, run the cell again and then the conversion cell (or *Run All*).

#### The results to convert

In [ ]:
# GAMS installation, used to read the gdx result files
gams_dir = 'C:/GAMS/46'

# folder with the results
input_dir = 'genesysmod_to_iamc/input'

# type of the results: 'gdx' (GAMS version) or 'csv' (Julia version)
file_type = 'gdx'

# the result file of every scenario, without extension (for csv results: the run name in the file names). The file
# names are also the scenario names in the outputs; to name them differently, write {file: name}, e.g.
# scenarios = {'GoRES_run1': 'Go RES', 'EUTrinity_run1': 'EU Trinity'}
scenarios = ['Go RES', 'REPowerEU++', 'NECP Essentials', 'EU Trinity']

# model years: the outputs cover only these years (other years in the results are left out)
years = [2018, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]

#### Names in the outputs

In [ ]:
# model name and release (scenario "Go RES v2.0.1", model "GENeSYS-MOD 4.1")
model = 'GENeSYS-MOD 4.1'
release = 'v2.0.1'

# regions: the sum over all model regions, and the sum over the EU member states in the model
total_region = 'Europe (incl. Turkey)'
eu27_region = 'EU27 (excl. Malta & Cyprus)'

#### What to convert

In [ ]:
transmission = True            # network capacities between regions (Network|...|Capacity)
hourly_series = False          # hourly demand series, in an extra file (large; needs Timeseries_<file>.xlsx)
load_factors = False           # hourly load factors of PV, wind and run-of-river, in an extra file (same)
combine = True                 # per scenario, one csv with the yearly values and the extra files
exogenous_costs = False        # switched off, kept for later use: capital, fixed and variable costs
additional_emissions = False   # switched off, kept for later use: CO2 emissions per technology

#### Outputs and the check before an upload

In [ ]:
# folder for the outputs
output_dir = 'genesysmod_to_iamc/out'

# name of the combined (upload) file, in the subfolder combined_excel of the output folder ({date} and {release}
# are filled in)
combined_file = 'GENeSYS-MOD-pathways_{date}.xlsx'

# variables left out of the upload file, e.g. while their definitions wait for a change in the explorer's workflow
# repository (* matches anything); the *_yearly.csv files keep them. Here: useful heat and the solar thermal and
# geothermal heat of buildings, whose iDesignRES definitions are not merged yet
leave_out = ['Useful Energy*',
             'Final Energy|Residential and Commercial|Solar',
             'Final Energy|Residential and Commercial|Geothermal']

# check the upload file with the upload workflow of the scenario explorer (usually
# https://github.com/iiasa/<project>-workflow; a copy of it is kept in ~/.genesysmod_to_iamc)
run_validation = True
workflow_repo = 'https://github.com/iiasa/idesignres-workflow'

### Conversion

The cell below passes the settings above to the converter and starts the conversion; nothing needs to be changed in it. It converts every scenario and writes `<scenario>_yearly.csv` (and `<scenario>_combined.csv`) to the output folder, combines the yearly values of all scenarios into the upload file in `combined_excel/`, and checks it with the explorer's workflow. The result of the check is in `validation_report.txt` in the output folder: `PASSED`, or the variables, regions and units that the explorer would reject. Four scenarios take about a quarter of an hour.

The explorer checks uploads with the newest `nomenclature-iamc`. To check with the same version, run `%pip install -U nomenclature-iamc pyam-iamc` in a cell before an upload and restart the kernel.

In [ ]:
# pass the settings to the converter
genesysmod_to_iamc.configure(
    gams_dir=gams_dir, input_dir=input_dir, file_type=file_type, scenarios=scenarios, years=years,
    model=model, release=release, total_region=total_region, eu27_region=eu27_region,
    transmission=transmission, hourly_series=hourly_series, load_factors=load_factors, combine=combine,
    exogenous_costs=exogenous_costs, additional_emissions=additional_emissions,
    output_dir=output_dir, combined_file=combined_file, leave_out=leave_out,
    run_validation=run_validation, workflow_repo=workflow_repo)

# convert, combine and check
genesysmod_to_iamc.run()

### Single steps

To convert only some of the scenarios, list them in `scenarios` above. The functions below use the settings passed in the conversion cell; to use them without converting, put `#` before `genesysmod_to_iamc.run()` there and run that cell first.

In [ ]:
# genesysmod_to_iamc.generate_data('Go RES')            # convert one scenario
# genesysmod_to_iamc.generate_combined_excel_yearly()   # combine all *_yearly.csv files in the output folder
# genesysmod_to_iamc.validate('genesysmod_to_iamc/out/combined_excel/EU-EnVis-2060_GENeSYS-MOD_pathways_v2.0.xlsx')